# 03b — Sequence tensors for the GRU (spec §3b)

Gross purchase-event sequences per (customer, cutoff) for the ≥2-purchase population. Key rules implemented:
**gap_since_prev is computed on the FULL ≤cutoff history BEFORE truncation** (never recomputed inside the truncated window); left padding + explicit boolean mask; static branch kept lean (tenure, is_uk, return summary). Event features: `[log1p(gap), log1p(basket_qty), log1p(basket_value), log1p(n_items), month_sin, month_cos, is_first_sequence_event]`.

Inputs: `RAW_PKL` (gross events rebuilt with the exact 03 filters) + `features_tabular.pkl` (population, label, split, static columns).
Output: `processed/sequences.npz` — `X_seq (n,L,7) float32, X_mask (n,L) bool, X_static (n,6), y, seq_len, customer_id, cutoff, split`. L=20 (≈P95 of sequence length; 05 can slice to shorter L).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, collections
from ds_common import *
INV="Invoice"; DT="InvoiceDate"; QTY="Quantity"; PR="Price"; SC="StockCode"
L=20
print("L =",L)

L = 20


## 1. Gross invoice events (identical filters to 03)

In [2]:
raw=pd.read_pickle(RAW_PKL)
raw=raw.drop_duplicates(subset=[c for c in raw.columns if c!="source_sheet"]).copy()
raw[DT]=raw[DT].astype("datetime64[ns]")
inv=raw[INV].astype(str); prod=raw[SC].astype(str).str.match(r"^\d{5}"); canc=inv.str.startswith("C")
g=raw[(~canc)&prod&(raw[QTY]>0)&(raw[PR]>0)&raw[KEY].notna()].copy(); g[KEY]=g[KEY].astype("int64")
g["line_val"]=g[QTY]*g[PR]
ev=(g.groupby([KEY,INV]).agg(event_time=(DT,"min"),gqty=(QTY,"sum"),gval=("line_val","sum"),
     n_items=(SC,"nunique")).reset_index().sort_values([KEY,"event_time"]).reset_index(drop=True))
print("gross events",len(ev),"| customers",ev[KEY].nunique())

gross events 36594 | customers 5852


## 2. Population = ≥2-purchase rows of the frozen panel (label/split/static reused, no recompute)

In [3]:
panel=load_features()
pop=panel[panel["frequency_total"]>=2][[KEY,"cutoff","split","churn","tenure_days","is_uk",
      "return_rate_value","days_since_last_return","fully_returned_order_rate","has_return_history"]].copy()
print("population rows:",len(pop),"| by split:",pop["split"].value_counts().to_dict(),
      "| churn:",round(pop["churn"].mean(),3))

loaded features_tabular.pkl: 47,933 rows x 39 cols | saved 2026-07-03 21:45
population rows: 35713 | by split: {'train': 16384, 'test': 11627, 'val': 7702} | churn: 0.433


## 3. Build tensors — gap on full ≤T history, then truncate to last L, left-pad + mask

In [4]:
Xs=np.zeros((len(pop),L,7),np.float32); Ms=np.zeros((len(pop),L),bool); SL=np.zeros(len(pop),np.int32)
row_pos={(int(k),T):i for i,(k,T) in enumerate(zip(pop[KEY],pop["cutoff"]))}   # (int, Timestamp) keys — stable across pandas/numpy versions
for T,sub in pop.groupby("cutoff"):
    custs=set(sub[KEY].values)
    e=ev[(ev["event_time"]<=T)&ev[KEY].isin(custs)].copy()
    e["gap"]=e.groupby(KEY)["event_time"].diff().dt.days.fillna(0.0)   # FULL-history gap, BEFORE truncation
    e["is_first"]=(e.groupby(KEY).cumcount()==0).astype(np.float32)
    mth=e["event_time"].dt.month.values
    feats=np.stack([np.log1p(e["gap"].values),np.log1p(e["gqty"].values),np.log1p(e["gval"].values),
                    np.log1p(e["n_items"].values),np.sin(2*np.pi*mth/12),np.cos(2*np.pi*mth/12),
                    e["is_first"].values],axis=1).astype(np.float32)
    per=collections.defaultdict(list)
    for j,k in enumerate(e[KEY].values): per[k].append(j)
    for k,js in per.items():
        i=row_pos[(int(k),T)]; take=js[-L:]; n=len(take); SL[i]=len(js)
        Xs[i,L-n:,:]=feats[take]; Ms[i,L-n:]=True
Xstat=pop[["tenure_days","is_uk","return_rate_value","days_since_last_return",
           "fully_returned_order_rate","has_return_history"]].values.astype(np.float32)
print("X_seq",Xs.shape,"| mask fill rate",round(Ms.mean(),3),
      "| seq_len P50/P75/P90/P95/max:",[int(np.percentile(SL,q)) for q in [50,75,90,95]],int(SL.max()))

X_seq (35713, 20, 7) | mask fill rate 0.305 | seq_len P50/P75/P90/P95/max: [4, 8, 13, 20] 278


## 4. Leakage / integrity assertions

In [5]:
assert (Ms.sum(1)==np.minimum(SL,L)).all(), "mask != min(seq_len,L)"
assert (SL>=2).all(), "single-purchase row slipped into the sequence population"
# spot-rebuild 3 rows incl. a truncated one: gap of the first kept slot must equal the TRUE previous-event gap
rng=np.random.default_rng(RANDOM_SEED)
for i in [int(np.where(SL>L)[0][0])]+list(rng.choice(np.where((SL>1)&(SL<=L))[0],2,replace=False)):
    k,T=pop[KEY].values[i],pop["cutoff"].values[i]
    e=ev[(ev[KEY]==k)&(ev["event_time"]<=T)].sort_values("event_time")
    gaps=np.log1p(e["event_time"].diff().dt.days.fillna(0.0).values[-min(SL[i],L):])
    assert np.allclose(gaps,Xs[i,Ms[i],0],atol=1e-5), f"gap-before-truncate violated at row {i}"
    assert (e["event_time"]<=T).all()
print("leakage/integrity assertions PASSED")

leakage/integrity assertions PASSED


## 5. Save

In [6]:
np.savez_compressed(PROC_DIR/"sequences.npz",X_seq=Xs,X_mask=Ms,X_static=Xstat,
    y=pop["churn"].values.astype(np.int8),seq_len=SL,
    customer_id=pop[KEY].values,cutoff=pop["cutoff"].values.astype("datetime64[ns]").astype("int64"),
    split=pop["split"].values.astype("U5"))
print("saved ->",(PROC_DIR/"sequences.npz").name)
print("static cols: tenure_days,is_uk,return_rate_value,days_since_last_return(NaN if never returned),fully_returned_order_rate,has_return_history")

saved -> sequences.npz
static cols: tenure_days,is_uk,return_rate_value,days_since_last_return(NaN if never returned),fully_returned_order_rate,has_return_history
